In [1]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import networkx as nx
import sspa

In [5]:
reactome_pathways = sspa.process_reactome(organism="Homo sapiens", download_latest=True)

Beginning Reactome download...
Complete!


In [2]:
def find_root(G,child):
    parent = list(G.predecessors(child))
    if len(parent) == 0:
        return child
    else:  
        return find_root(G, parent[0])
    
# load the pathway database file from the data folder

hierarchy = pd.read_csv('/home/scostagonza/Documents/Pathway_hierarchy_rel.txt', sep='\t', header=None)
hierarchy_hsa = hierarchy[hierarchy[0].str.contains('HSA')]
hierarchy_hsa_parents = np.setdiff1d(hierarchy_hsa[0], hierarchy_hsa[1])
hierarchy_hsa_all = pd.concat([hierarchy_hsa, pd.DataFrame([hierarchy_hsa_parents, hierarchy_hsa_parents], index=[0, 1]).T])

# the default graph is the pathway hierarchy coloured by root pathway membership as defined by Reactome
G = nx.from_pandas_edgelist(hierarchy_hsa, source=0, target=1, create_using=nx.DiGraph())

In [3]:
# the default graph is the pathway hierarchy coloured by root pathway membership as defined by Reactome
G = nx.from_pandas_edgelist(hierarchy_hsa, source=0, target=1, create_using=nx.DiGraph())
hierarchy_hsa_all['Root'] = [find_root(G, i) for i in hierarchy_hsa_all[1]]
root_cmap = dict(zip(set(hierarchy_hsa_all['Root']), sns.color_palette("husl", len(set(hierarchy_hsa_all['Root']))).as_hex()))

In [6]:
name_dict = dict(zip(reactome_pathways.index, reactome_pathways['Pathway_name']))
G.add_nodes_from([(node, {'Name': attr, 'label': attr}) for (node, attr) in name_dict.items()])

In [7]:
G.add_nodes_from([(node, {'Root': attr, 
                              'RootCol': root_cmap[attr], 
                              'color': root_cmap[attr], 
                              'RootName': name_dict[attr]}) for (node, attr) in dict(zip(hierarchy_hsa_all[1], hierarchy_hsa_all['Root'])).items()])

In [8]:
nx.write_graphml(G, '/home/scostagonza/Documents/Pathway_hierarchy.graphml')